# Loan clusterization

In this notebook, we will analyse the German credit dataset, which was originally created by Hans Hofmann in 1994, and contains registries of loans made in a bank in German at that time. Our goal is to **cluster profiles of loans in the bank**. The dataset was obtained in https://www.kaggle.com/datasets/uciml/german-credit on 2026-08-05.

## Importing libraries and data loading

These are the libraries we will use:

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler

We will load the CSV file with all the data:

In [ ]:
df = pd.read_csv('german_credit_data.csv')

## Exploratory Data Analysis (EDA)

### Data cleaning

We see if there are any duplicated data:

In [ ]:
df.duplicated().any()

There are **no duplicated data**. Let's see if there are null values:

In [ ]:
df.info()

As we can see, there are **1000 entries** (a good amount), but there are 183 people with empty registry on saving accounts and 394 people with the same situation on checking accounts. This is a very significative part of the entire dataset (18.3% and 39.4%). Plus, there's a column, named 'unnamed', representing the IDs, that can be ignored. So, we run this command to discard that specific column:

In [ ]:
df = df.drop(columns=['Unnamed: 0'])

And we take a look at the possible data on the saving accounts and checking accounts columns:

In [ ]:
print(df['Saving accounts'].value_counts(dropna=False))
print("\n")
print(df['Checking account'].value_counts(dropna=False))

So, we could interpret **'NaN' simply as the client does not have that kind of account in the bank**, and the other adjectives, such as 'moderate' and 'rich', describe, without numbers, the amount of money in the clients' accounts. However, let's look into the existence of people without both accounts:

In [ ]:
len(df[df['Saving accounts'].isna() & df['Checking account'].isna()])

This number is very expressive, as it's approximately 10% of the number of entries in the dataset. Therefore, we simply cannot drop data, we treat the absence of info on the saving accounts and checking accounts as lack of these kinds of accounts in the bank. It means the person is asking for a loan without having account in the bank in which he/she is asking for. So, the person can be a client of other bank too. Let's consider the lack of info as **'no account'**:

In [ ]:
df['Saving accounts'] = df['Saving accounts'].fillna('no account')
df['Checking account'] = df['Checking account'].fillna('no account')

### Univariate analysis

#### Analyzing age

In [ ]:
print(df['Age'].describe())
print("\n")

In [ ]:
sns.histplot(data=df, x='Age', kde=True)
plt.show()

sns.boxplot(data=df, y='Age')
plt.show()

This reveals us that the common profile of the loans is made by **young** clients.

#### Analyzing sex

In [ ]:
print(df['Sex'].value_counts())
print("\n")

In [ ]:
sns.countplot(data=df, x='Sex')
plt.show()

69% of the loans were made by men and 31% by women. This shows us that the profile of the common loan has a **masculine** source, but there is a significative feminine presence in the dataset.

#### Analyzing job

In [ ]:
print(df['Job'].value_counts())
print("\n")

In [ ]:
sns.countplot(data=df, x='Job')
plt.show()

In the dataset, the value 0 for job means 'unskilled and non-resident', 1 'unskilled and resident', 2 'skilled' and 3 'highly skilled'. Therefore, most of the clients of the bank are **skilled or highly skilled (77.8%)**.

#### Analyzing housing

In [ ]:
print(df['Housing'].value_counts())
print("\n")

In [ ]:
sns.countplot(data=df, x='Housing', order=df['Housing'].value_counts().index)
plt.show()

Most clients have their **own home (71.3%)**, and the numbers of people that live in a rented place and in a spot that are not theirs, but they don't pay for living in that location, are close. 

#### Analyzing saving accounts

In [ ]:
print(df['Saving accounts'].value_counts())
print("\n")

In [ ]:
sns.countplot(data=df, x='Saving accounts', order=df['Saving accounts'].value_counts().index)
plt.show()

Most clients have **little or no saving accounts (78.6%)**. Just a few (11.1%) are rich or quite rich, which is a little greater than people with moderate money in saving accounts (10.3%).

#### Analyzing checking account

In [ ]:
print(df['Checking account'].value_counts())
print("\n")

In [ ]:
sns.countplot(data=df, x='Checking account', order=df['Checking account'].value_counts().index)
plt.show()

Regarding checking account, **most clients (66.8%) don't have it or have just a small amount of money**. The number of 'moderate' is approximately the number of 'little', and 'rich' is not common.

#### Analyzing credit amount

In [ ]:
print(df['Credit amount'].describe())
print("\n")

In [ ]:
sns.histplot(data=df, x='Credit amount', kde=True)
plt.show()

sns.boxplot(data=df, y='Credit amount')
plt.show()

Credit amount is given in DM (Deutsch Mark). There are **many outliers with credits above 7500**, when 75% of the loans are less than 3972.25. We notice nested outliers.

#### Analyzing duration

In [ ]:
print(df['Duration'].describe())
print("\n")

In [ ]:
sns.histplot(data=df, x='Duration', kde=True)
plt.show()

sns.boxplot(data=df, y='Duration')
plt.show()

Duration is given in months. Again, we notice the presence of outliers, but they come in less number comparing to the credit amount. And we see peaks at 5 regions.

#### Analyzing purpose

In [ ]:
print(df['Purpose'].value_counts())

In [ ]:
sns.countplot(data=df, x='Purpose', order=df['Purpose'].value_counts().index)
plt.xticks(rotation=45, ha='right')
plt.show()

The purposes of the loans are **diverse**. Some of them require naturally more duration and credit amount, such as 'car' and 'business', while others, as 'radio/TV', don't.

### Multivariate analysis

#### Analyzing credit amount, duration and purpose

In [ ]:
print("Considering credit amount and purpose:\n\n")
df.groupby('Purpose')['Credit amount'].describe()

Vacation/others has **very huge numbers** compared to other purposes, which tend to be closer to each other.

In [ ]:
print("Considering duration and purpose:\n\n")
df.groupby('Purpose')['Duration'].describe()

When **we descard credit amount and consider only duration**, all purposes, including vacation/others, tend to be closer to each other.

In [ ]:
def plot_scatter(x, y, **kwargs):
    sns.scatterplot(x=x, y=y)
    sns.regplot(
        x=x,
        y=y,
        scatter=False,
        color="red",
        ci=None,
        line_kws={"linewidth": 2},
    )

g = sns.FacetGrid(df, col="Purpose", col_wrap=2, sharex=False, sharey=False)
g.map(plot_scatter, "Credit amount", "Duration")

for i in g.axes.flat:
    i.set_xlabel("Credit amount")
    i.set_ylabel("Duration")

g.fig.subplots_adjust(hspace=0.6, wspace=0.6)

plt.show()

With these data, we can infer that:
 - The durations tend to grow when the credit amounts goes up.
 - The rate of this growing depends on the the kind of purpose.

## Data preprocessing

Now, in order to cluster, we need to turn some qualitative variables into quantitative ones. So, we use **One-Hot Encoding** and **Ordinal Encoding**.

Job is already encoded in the dataset.

In [ ]:
df = pd.get_dummies(df, columns=['Sex', 'Purpose'], drop_first=True)

df['Saving accounts'] = df['Saving accounts'].map({'no account': 0, 'little': 1, 'moderate': 2, 'rich': 3, 'quite rich': 4})
df['Checking account'] = df['Checking account'].map({'no account': 0, 'little': 1, 'moderate': 2, 'rich': 3})
df['Housing'] = df['Housing'].map({'rent': 1, 'free': 2, 'own': 3})

And we need to standard our data. So, we use **StandardScaler**.

In [ ]:
df[['Age', 'Credit amount', 'Duration']] = StandardScaler().fit_transform(df[['Age', 'Credit amount', 'Duration']])

Let's take a brief look at our dataframe:

In [ ]:
df.head()